In [1]:
import numpy as np

In [2]:
tags = ['O', 'ART', 'N', 'V', 'P']
tag_to_idx = {tag: i for i, tag in enumerate(tags)}

In [3]:
# P(Ci | Ci-1)
trans_prob = np.array([
    [0, 0.71, 0.29, 0, 0],    # O -> [ART: 0.71, N: 0.29]
    [0, 0, 1.0, 0, 0],       # ART -> [N: 1.0]
    [0, 0, 0.13, 0.43, 0.44], # N -> [N: 0.13, V: 0.43, P: 0.44]
    [0, 0.65, 0.35, 0, 0],    # V -> [ART: 0.65, N: 0.35]
    [0, 0.74, 0.26, 0, 0]     # P -> [ART: 0.74, N: 0.26]
])

In [4]:
# 2. Dữ liệu xác suất từ vựng từ Hình 3.4
# Giả sử "flowers" (số nhiều) có xác suất tương tự "flower" hoặc "birds" (N) trong tài liệu
vocab_prob = {
    "flower": {"N": 0.063, "V": 0.05},
    "flowers": {"N": 0.076}, # Giả định theo birds/N
    "like": {"V": 0.1, "P": 0.068, "N": 0.12}
}

In [5]:
words = ["flower", "flowers", "like", "flowers"]
T = len(words)
N = len(tags)

# Khởi tạo bảng Viterbi
viterbi = np.zeros((N, T))
backpointer = np.zeros((N, T), dtype=int)

In [7]:
# Bước khởi tạo (t=0)
for i in range(1, N):
    tag = tags[i]
    p_emission = vocab_prob[words[0]].get(tag, 0.0001)
    viterbi[i, 0] = trans_prob[tag_to_idx['O'], i] * p_emission

# Bước lặp (t=1 -> T-1)
for t in range(1, T):
    for i in range(1, N):
        tag = tags[i]
        p_emission = vocab_prob[words[t]].get(tag, 0.0001)
        
        # Tìm max (viterbi[j, t-1] * trans_prob[j, i])
        probs = [viterbi[j, t-1] * trans_prob[j, i] for j in range(1, N)]
        viterbi[i, t] = max(probs) * p_emission
        backpointer[i, t] = np.argmax(probs) + 1

In [8]:
# Bước xác định chuỗi (Backtracking)
best_path = [0] * T
best_path[T-1] = np.argmax(viterbi[:, T-1])

In [9]:
for t in range(T-2, -1, -1):
    best_path[t] = backpointer[best_path[t+1], t+1]

print(f"Câu: {' '.join(words)}")
print(f"Chuỗi từ loại tối ưu: {' -> '.join([tags[i] for i in best_path])}")

Câu: flower flowers like flowers
Chuỗi từ loại tối ưu: N -> N -> V -> N


In [10]:
from collections import Counter

In [11]:
corpus = [
    "<s> I am Sam </s>",
    "<s> Sam I am </s>",
    "<s> I am Sam </s>",
    "<s> I do not like green eggs and Sam </s>"
]

# Tách từ (tokens)
all_tokens = []
bigrams = []

In [ ]:
for sentence in corpus:
    words = sentence.split()
    all_tokens.extend(words)
    for i in range(len(words) - 1):
        bigrams.append((words[i], words[i+1]))

# Tính toán các số đếm
vocab = set(all_tokens)
V = len(vocab) # Kích thước từ vựng
unigram_counts = Counter(all_tokens)
bigram_counts = Counter(bigrams)

# Tính P(Sam | am) với làm mịn cộng một
# Công thức: P(w_n | w_n-1) = (C(w_n-1, w_n) + 1) / (C(w_n-1) + V)
count_am_sam = bigram_counts[('am', 'Sam')]
count_am = unigram_counts['am']

p_sam_am = (count_am_sam + 1) / (count_am + V)

In [ ]:
print(f"Số lần bigram ('am', 'Sam') xuất hiện: {count_am_sam}")
print(f"Số lần unigram 'am' xuất hiện: {count_am}")
print(f"Kích thước từ vựng V: {V}")P
print(f"P(Sam | am) với Add-one smoothing = ({count_am_sam} + 1) / ({count_am} + {V}) = {p_sam_am:.4f}")

Số lần bigram ('am', 'Sam') xuất hiện: 2
Số lần unigram 'am' xuất hiện: 3
Kích thước từ vựng V: 11
P(Sam | am) với Add-one smoothing = (2 + 1) / (3 + 11) = 0.2143
